# 5회차 | 전처리(Preprocessing)

**핵심 질문**: 데이터를 그대로 넣으면 안 되는 이유는?

**오늘의 목표**
1. **인코딩**: 글자(문자열)는 모델이 이해 못 한다 → 숫자로 바꿔주기
   - Label vs One-Hot vs Ordinal — 언제 뭘 쓰는가
2. **스케일링**: 값의 크기가 다르면 공정한 비교가 안 된다 → 크기 맞춰주기
   - StandardScaler vs MinMaxScaler vs RobustScaler
3. **정규화**: 각 데이터의 길이를 똑같이 맞춰주는 방법 (방향만 중요할 때)
4. **데이터 누수**: 시험지를 미리 보면 안 된다 → 훈련 데이터로만 fit → Pipeline으로 안전하게


---
## 오늘 전처리를 다루는 이유

**지난 시간에 모델을 선택하는 기준을 배웠음**
하지만 좋은 모델을 골라도, **데이터가 제대로 준비되지 않으면 성능이 안 나옴**

![최고의 셰프도 상한 재료로는 요리할 수 없습니다](스크린샷%202026-02-08%20오전%209.40.41.png)

![모델을 위한 재료 손질: 전처리](스크린샷%202026-02-08%20오전%209.41.06.png)

### 전처리가 필요한 이유

| 문제 | 결과 | 해결책 |
|------|------|--------|
| 문자열 데이터 | 모델이 계산 불가 | **인코딩** |
| 값의 크기 차이(단위/범위) | 특정 피처가 지배 | **스케일링** |
| 길이(크기)보다 방향이 중요 | 의미/패턴 비교가 어려움 | **정규화** |
| 전처리를 전체 데이터로 fit | 평가가 무효(누수) | **Pipeline** |

> **3회차 핵심 문장:**
> "모델은 숫자만 이해하고, 크기가 비슷해야 공정하게 학습함."

---
## Part 1. 인코딩 (Encoding)

### 인코딩이란?

- 모델은 글자(문자열)를 직접 계산할 수 없음
- 예: 색깔 = "빨강(red)", "초록(green)", "파랑(blue)"
- 글자를 그대로 넣으면 모델은 "이게 뭐지?" 하고 멈춤

> 마치 한국어 시험지를 원어민 영어 선생님에게 가져가면, 선생님은 못 알아보는 것과 같음
> 그래서 "숫자"로 번역(인코딩)을 해줘야 함

In [1]:
import pandas as pd

data = pd.DataFrame({"color": ["red","green","blue","red","green","blue"]})
data

,color
0,red
1,green
2,blue
3,red
4,green
5,blue


### 레이블 인코딩 vs 원-핫 인코딩

1) **레이블 인코딩(Label Encoding)**
   - red=0, green=1, blue=2 처럼 번호 붙이기
   - 문제: 0, 1, 2 이런식으로 하면 숫자의 크기가 반영, 크기로 인해서 모델이 오해(착시) 0<1<2

2) **원-핫 인코딩(One-Hot Encoding)**
   - red = [1,0,0]
   - green = [0,1,0]
   - blue = [0,0,1]
   - 순서를 만들지 않고, 단지 "있다/없다"만 표시

### 참고: 차원(Dimension)이 뭔가요?

- **차원 수**란, 모델이 한 데이터를 판단할 때 **참고하는 입력값의 개수**
- 실습에서는 보통 **"모델에 들어가는 컬럼의 개수"**라고 이해하면 됨

> **예시**
> - 원래 color 1개 컬럼 → 차원 = 1
> - 원-핫 인코딩 후 red, green, blue 3개 컬럼 → 차원 = 3

- 원-핫 인코딩처럼 전처리를 하면 모델이 데이터를 더 잘 이해할 수 있도록 **차원이 늘어날 수 있음**
- 하지만 차원이 늘수록 **데이터 구조는 더 복잡해지고**, 중요한 정보와 중요하지 않은 정보가 섞일 수 있음
- 그래서 이후에는 **차원을 줄이거나 중요한 축만 남기는 방법**도 필요해짐 (나중에 배움)

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 레이블 인코딩
le = LabelEncoder()
data["color_label"] = le.fit_transform(data["color"])

# 원-핫 인코딩
ohe = OneHotEncoder(sparse_output=False)
ohe_matrix = ohe.fit_transform(data[["color"]])
ohe_df = pd.DataFrame(ohe_matrix, columns=ohe.categories_[0])

data_ohe = pd.concat([data, ohe_df], axis=1)
data_ohe

,color,color_label,blue,green,red
0,red,2,0.0,0.0,1.0
1,green,1,0.0,1.0,0.0
2,blue,0,1.0,0.0,0.0
3,red,2,0.0,0.0,1.0
4,green,1,0.0,1.0,0.0
5,blue,0,1.0,0.0,0.0


- "color_label" 열은 0,1,2로 번호를 붙인 것 (순서 착각 위험)
- "blue, green, red" 열은 원-핫 인코딩 결과 (순서 없음, 안전)

> 실제로는 **원-핫 인코딩**을 많이 씁니다.

### 레이블 인코딩의 문제점 — 선형 모델에서 확인(모델은 나중에 배울 예정이니 그런가보다 하셈)

레이블 인코딩은 **트리 모델(나중에 배움)**에서는 괜찮지만,
**선형/거리 기반 모델**에서는 "가짜 순서"가 학습에 영향을 줄 수 있음.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import numpy as np
import pandas as pd

np.random.seed(42)
colors = pd.Series(np.random.choice(["red", "green", "blue"], size=100))

target = colors.map({
    "red": 1,
    "blue": 1,
    "green": 0
})

df_test = pd.DataFrame({
    "color": colors,
    "target": target
})

# 레이블 인코딩
X_label = LabelEncoder().fit_transform(df_test["color"]).reshape(-1,1)

# 원-핫 인코딩
X_onehot = OneHotEncoder(sparse_output=False).fit_transform(df_test[["color"]])

y = df_test["target"]

print("=== 로지스틱 회귀 (선형 모델) ===")
score_label = cross_val_score(LogisticRegression(), X_label, y, cv=5).mean()
score_onehot = cross_val_score(LogisticRegression(), X_onehot, y, cv=5).mean()
print(f"레이블 인코딩: {score_label:.3f}")
print(f"원-핫 인코딩: {score_onehot:.3f}")
# === 로지스틱 회귀 (선형 모델) ===
# 레이블 인코딩: 0.640
# 원-핫 인코딩: 1.000

print("\n=== 결정 트리 (트리 모델) ===")
score_label_dt = cross_val_score(DecisionTreeClassifier(random_state=42), X_label, y, cv=5).mean()
score_onehot_dt = cross_val_score(DecisionTreeClassifier(random_state=42), X_onehot, y, cv=5).mean()
print(f"레이블 인코딩: {score_label_dt:.3f}")
print(f"원-핫 인코딩: {score_onehot_dt:.3f}")
# === 결정 트리 (트리 모델) ===
# 레이블 인코딩: 1.000
# 원-핫 인코딩: 1.000

print("\n→ 트리 모델은 인코딩 방식에 덜 민감하지만,")
print("  선형 모델은 원-핫 인코딩이 더 안전함")

=== 로지스틱 회귀 (선형 모델) ===
레이블 인코딩: 0.640
원-핫 인코딩: 1.000

=== 결정 트리 (트리 모델) ===
레이블 인코딩: 1.000
원-핫 인코딩: 1.000

→ 트리 모델은 인코딩 방식에 덜 민감하지만,
  선형 모델은 원-핫 인코딩이 더 안전함


### Ordinal(순서형) 인코딩

- **명목형(순서 없음)**: 원-핫 인코딩
- **순서형(작<중<대)**: OrdinalEncoder(순서 지정) 가 가장 안전

> 순서가 **의미 있는** 경우에만 오디널 인코딩을 사용!

In [4]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df_ord = pd.DataFrame({"size":["small","medium","large","medium","small"]})
ord_enc = OrdinalEncoder(categories=[["small","medium","large"]])
df_ord["size_code"] = ord_enc.fit_transform(df_ord[["size"]])
df_ord

,size,size_code
0,small,0.0
1,medium,1.0
2,large,2.0
3,medium,1.0
4,small,0.0


### 인코딩 방법 비교

| 방법 | 설명 | 언제 쓰나 | 주의점 |
|------|------|----------|--------|
| **레이블 인코딩** | red=0, green=1, blue=2 | (주로) **타깃 y** 라벨, 또는 **순서형**을 임시로 코드화 | X에 쓰면 **가짜 순서** 위험 (특히 선형/거리 모델) |
| **원-핫 인코딩** | red=[1,0,0], green=[0,1,0] | **순서 없는 명목형 X** (가장 안전한 기본값) | 차원 증가(카테고리 많으면 폭발) |
| **오디널 인코딩** | small=0, medium=1, large=2 | **순서가 의미 있을 때** | 순서를 직접 지정해야 함 |

### 인코딩 선택 가이드(항상 예외는 있음)

```markdown
범주형 데이터인가?
    ├── 순서가 있는가? (small < medium < large)
    │       └── YES → OrdinalEncoder(※ 순서 자체가 의미일 때만)
    │       └── NO  → 아래로
    └── 어떤 모델을 쓰는가?
            ├── 트리 모델 (DecisionTree, RandomForest, XGBoost)
            │       └── LabelEncoder도 OK (**One-Hot** 권장 / 순서형이면 **Ordinal**)
            └── 선형/거리 기반 (LogisticRegression, KNN, SVM)
                    └── OneHotEncoder 필수!
```

### 텍스트 인코딩 — CountVectorizer로 문장을 숫자로 바꾸기

지금까지 색상(red, green, blue)처럼 **짧은 범주형** 데이터를 인코딩했음
그렇다면 **문장(텍스트)**은 어떻게 숫자로 바꿀까??????

- **CountVectorizer**: 문장에서 단어가 **몇 번 나왔는지(빈도)**를 세어 벡터로 변환
- 예: "영화 재밌다" → [1, 0, 1, 0, ...] (단어별 등장 횟수)

> 이것도 일종의 인코딩임!
> 글자를 숫자로 바꿔서 모델이 계산할 수 있게 만드는 것.

> 참고: 한국어는 공백 기준 토큰화만으로는 단어가 부정확할 수 있음.
> (실무에서는 형태소 분석기나 서브워드 토크나이저를 쓰기도 함.)

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# 문장 리스트 = 우리가 가진 텍스트 데이터
texts = [
    "이 영화 정말 재밌다",
    "완전 감동적이다",
    "너무 지루하고 별로였다",
    "시간 낭비였다",
    "연출이 좋았다",
    "스토리가 엉망이다"
]

# 라벨: 1 = 긍정, 0 = 부정
labels = [1, 1, 0, 0, 1, 0]

# 2) 벡터화: 문장 → 숫자 배열 
# CountVectorizer = 단어가 몇 번 나왔는지(빈도)를 세는 도구
vectorizer = CountVectorizer()

# fit_transform: 단어 사전을 만들고(fit) + 문장을 숫자로 변환(transform)
X = vectorizer.fit_transform(texts)

# 어떤 단어들이 사전에 들어갔는지 확인
print("단어 사전:", vectorizer.get_feature_names_out())
# ['감동적이다' '낭비였다' '너무' '별로였다' '스토리가' '시간' '엉망이다' '연출이' '영화' '완전' '재밌다' '정말' '좋았다' '지루하고']

# 3) 모델 학습 
# LogisticRegression = 확률 기반 분류기 (나중에 배움)
model = LogisticRegression()
model.fit(X, labels)
# 모델이 학습한 것: 이 단어 조합이면 긍정일 확률이 높구나

# 4) 새 문장 예측 
new_text = ["이 영화는 별로다"]

# 주의: 새 문장은 transform만! (fit 하면 안 됨 — 사전이 바뀌니까)
new_X = vectorizer.transform(new_text)

prediction = model.predict(new_X)
print("예측 결과:", prediction) # [0]

단어 사전: ['감동적이다' '낭비였다' '너무' '별로였다' '스토리가' '시간' '엉망이다' '연출이' '영화' '완전' '재밌다' '정말'
 '좋았다' '지루하고']
예측 결과: [0]


---
## Part 2. 스케일링 (Scaling)

### 스케일링이란?

- 두 가지 정보가 있습니다.
  - 학생 키 = [150cm, 160cm, 170cm] → 범위: 20
  - 학생 용돈 = [1000원, 20000원, 50000원] → 범위: 49000

> 용돈 값이 너무 커서 모델은 키를 무시하게 됨
> 마치 수학 점수는 0-100, 음악 점수는 0-1억일 때 → 음악만 중요해지는 상황
> 그래서 범위를 비슷하게 맞춰주는 작업이 필요

In [6]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.array([[150,1000],
              [160,20000],
              [170,50000]])

scaler = StandardScaler()
# 평균 0, 표준편차 1 (-1 ~ 1)
# 모델이 공평하게 학습(비교도 하구)
X_scaled = scaler.fit_transform(X)

df_scale = pd.DataFrame(
    np.hstack([X, X_scaled]),
    columns=["키(cm)","용돈(원)","키(스케일링 후)","용돈(스케일링 후)"]
)
df_scale

# 	키(cm)	용돈(원)	키(스케일링 후)	용돈(스케일링 후)
# 0	150.0	1000.0	-1.224745	-1.123698
# 1	160.0	20000.0	0.000000	-0.181775
# 2	170.0	50000.0	1.224745	1.305473


,키(cm),용돈(원),키(스케일링 후),용돈(스케일링 후)
0,150.0,1000.0,-1.224745,-1.123698
1,160.0,20000.0,0.000000,-0.181775
2,170.0,50000.0,1.224745,1.305473


- 원래 데이터는 "용돈" 값이 너무 커서 차이가 엄청 큼
- 스케일링 후에는 "키"와 "용돈"이 비슷한 범위로 맞춰짐

> 이제 두 특성이 공정하게 반영될 수 있음

### RobustScaler — 이상치에 강한 스케일러
- 중앙값/사분위 범위(IQR)로 스케일링 → 평균/표준편차 대신 **통계량** 사용

In [7]:
import numpy as np, pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler

X = np.array([[1., 10.],
              [2., 12.],
              [3., 13.],
              [100., 1000.]])   # 이상치
# 평균 = 0, 표준편차 1로 변환 (z-Score)
# 평균을 가지고 노니까 이상치에 민감
sc = StandardScaler().fit_transform(X)
# 중앙값, IQR을 가지고 스케일링
# 이상치에 상대적으로 덜 민감함
rb = RobustScaler().fit_transform(X)

df_rb = pd.DataFrame(np.hstack([X, sc, rb]),
                     columns=["x1","x2","Std_x1","Std_x2","Robust_x1","Robust_x2"])
df_rb

# 	x1	x2	Std_x1	Std_x2	Robust_x1	Robust_x2
# 0	1.0	10.0	-0.600832	-0.581243	-0.058824	-0.010070
# 1	2.0	12.0	-0.577270	-0.576570	-0.019608	-0.002014
# 2	3.0	13.0	-0.553708	-0.574233	0.019608	0.002014
# 3	100.0	1000.0	1.731810	1.732045	3.823529	3.977845

,x1,x2,Std_x1,Std_x2,Robust_x1,Robust_x2
0,1.0,10.0,-0.600832,-0.581243,-0.058824,-0.010070
1,2.0,12.0,-0.577270,-0.576570,-0.019608,-0.002014
2,3.0,13.0,-0.553708,-0.574233,0.019608,0.002014
3,100.0,1000.0,1.731810,1.732045,3.823529,3.977845


### MinMaxScaler vs StandardScaler
- MinMax: [0,1] 구간으로 맞춤 (최솟값/최댓값 기반)
- Standard: 평균 0, 표준편차 1 (z-score 표준화)
- 어떤 스케일러가 좋은지는 **데이터 분포/모델**에 따라 다름

In [8]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y, random_state=42)


pipe_minmax = Pipeline([("scaler", MinMaxScaler()),
                        ("knn", KNeighborsClassifier(n_neighbors=5))])
pipe_std    = Pipeline([("scaler", StandardScaler()),
                        ("knn", KNeighborsClassifier(n_neighbors=5))])

pipe_minmax.fit(X_train, y_train); pred_mm = pipe_minmax.predict(X_test)
pipe_std.fit(X_train, y_train);    pred_sd = pipe_std.predict(X_test)

print(f"[MinMax] Acc={accuracy_score(y_test, pred_mm):.3f}, F1(macro)={f1_score(y_test, pred_mm, average='macro'):.3f}")
print(f"[Std   ] Acc={accuracy_score(y_test, pred_sd):.3f}, F1(macro)={f1_score(y_test, pred_sd, average='macro'):.3f}")

# [MinMax] Acc=0.933, F1(macro)=0.933
# [Std   ] Acc=0.911, F1(macro)=0.910

[MinMax] Acc=0.933, F1(macro)=0.933
[Std   ] Acc=0.911, F1(macro)=0.910


> **참고: Iris + KNN에서 MinMaxScaler가 잘 작동한 이유**
>
> KNN은 거리 기반 모델로, 피처 간 상대적 거리 구조가 중요 
> Iris 데이터는 petal length, petal width 같은 피처에서 클래스가 **값의 범위(range)**로 명확히 분리되어 있음
>
> 예: petal length 단순화
> - Class A: 1.0 ~ 2.0
> - Class B: 3.0 ~ 4.0  
> - Class C: 5.0 ~ 6.0
>
> MinMaxScaler 적용 후 (min=1, max=6 기준):
> - Class A: 0.00 ~ 0.20
> - Class B: 0.40 ~ 0.60
> - Class C: 0.80 ~ 1.00
>
> → 클래스 간 간격은 유지되고, 모든 피처가 동일한 스케일을 가지게 되어 거리 계산에 유리해짐.  
> → MinMaxScaler는 값을 섞는 것이 아니라, **거리 구조를 보존한 채 스케일만 통일**하는 역할을 함.


### 핵심
- 거리 기반(KNN) → 스케일링 거의 필수
- 데이터가 깔끔하고 범위 구조가 중요 → MinMax가 잘 먹히는 경우 많음

### 스케일러 비교

| 스케일러 | 방식 | 특징 | 언제 쓰나 |
|---------|------|------|----------|
| **StandardScaler** | 평균=0, 표준편차=1 (z-score) | 많은 모델에서 안정적으로 잘 작동 | 기본 선택(선형/거리 기반에서 자주) |
| **MinMaxScaler** | [0, 1] 구간으로 변환 | 이상치(outlier)에 민감 | 범위가 중요한 값(픽셀 등) / 데이터가 깔끔할 때 |
| **RobustScaler** | 중앙값/IQR 기반 | 이상치에 강함 | 이상치가 있을 때 |

### 스케일러 선택 가이드(항상 예외는 있음)

```
데이터에 이상치가 있는가?
    ├── YES → RobustScaler
    └── NO  → 아래로
            ├── 값의 범위가 중요한가? (예: 이미지 픽셀 0~255)
            │       └── YES → MinMaxScaler
            └── NO  → StandardScaler (가장 일반적)
```

---
## Part 3. 정규화 (Normalization)

### 정규화란?
- 앞에서 CountVectorizer로 만든 것도 전부 **벡터(숫자 배열)**이었음
- 이제부터는 벡터를 이렇게 해석합세:
  - **길이(크기)**: “얼마나 많이/크게” (예: 문장 길이, 단어 총량)
  - **방향(패턴)**: “어떤 조합/비율인가” (예: 어떤 단어들이 함께 나왔는가)

> 정규화의 목적: **길이 영향은 지우고, 방향(패턴)만 비교**

정규화는 각 데이터 벡터의 길이를 1로 맞추는 것.  
예: [3,4] → 길이=5 → [0.6,0.8]

> 마치 지도에서 "집까지의 거리"는 무시하고, "집이 어느 방향에 있는지"만 보는 것과 같음

**음악 취향 비유:**
- A는 록 100곡+재즈 50곡, B는 록 10곡+재즈 5곡
- → 절대 개수(길이)는 다르지만, 비율(방향)은 동일
- → 정규화하면 같은 취향 패턴으로 인식됨


#### 정리
- 스케일링은 **피처(열) 기준**, 정규화는 **샘플(행) 기준**
- 정규화는 **방향 비교**가 핵심 (단어/추천/임베딩에서 자주 사용)

In [9]:
from sklearn.preprocessing import Normalizer

X = np.array([[3,4],
              [1,2],
              [10,0]])

normalizer = Normalizer()
X_norm = normalizer.fit_transform(X)

df_norm = pd.DataFrame(
    np.hstack([X, X_norm]),
    columns=["x1","x2","x1(정규화 후)","x2(정규화 후)"]
)
df_norm

# 	x1	x2	x1(정규화 후)	x2(정규화 후)
# 0	3.0	4.0	0.600000	0.800000
# 1	1.0	2.0	0.447214	0.894427
# 2	10.0	0.0	1.000000	0.000000


,x1,x2,x1(정규화 후),x2(정규화 후)
0,3.0,4.0,0.600000,0.800000
1,1.0,2.0,0.447214,0.894427
2,10.0,0.0,1.000000,0.000000


### 정규화와 코사인 유사도

- **코사인 유사도(cosine similarity)**는 두 벡터의 **방향(각도)**이 얼마나 비슷한지 보는 지표임
  - 같은 방향이면 1
  - 직각(관련 거의 없음)에 가까우면 0
  - 반대 방향이면 -1

- 벡터를 **L2 정규화**하면 길이가 1이 됨
- 이때는 **코사인 유사도 = 내적(dot product)**이 됨
  - (길이가 1이라서 분모가 사라지는 효과)

> 그래서 텍스트 벡터/임베딩 비교에서
> **정규화 + 내적(=코사인)** 조합이 자주 등장함

In [10]:
import numpy as np
from sklearn.preprocessing import Normalizer
from numpy.linalg import norm

A = np.array([[3.,4.]])
B = np.array([[6.,8.]])

normer = Normalizer()
A_n = normer.fit_transform(A); B_n = normer.transform(B)

cos = (A @ B.T) / (norm(A) * norm(B))
dot_after_norm = (A_n @ B_n.T)
print(f"cosine(A,B)= {cos.item()}")
print(f"dot(normalized)= {dot_after_norm.item()}")

# cosine(A,B)= 1.0
# dot(normalized)= 1.0


cosine(A,B)= 1.0
dot(normalized)= 1.0


### 문장 유사도 — CountVectorizer + 정규화 + 코사인 유사도

앞에서 배운 개념들을 조합하면 **문장이 얼마나 비슷한지**도 계산할 수 있음.

1. **CountVectorizer**: 문장 → 단어 빈도 벡터
2. **Normalizer**: 벡터 길이를 1로 맞춤 (방향만 남김)
3. **cosine_similarity**: 두 벡터의 방향이 같으면 1, 다르면 0에 가까움

> CountVectorizer는 문장을 숫자 벡터로 바꾸고,  
> 정규화와 코사인 유사도는 그 벡터의 **방향(의미)**이 같은지를 봄.

> **LLM의 임베딩도 결국 같은 아이디어**
> (문장을 벡터로 만들고, 벡터의 방향 유사도를 비교)
> 다만 CountVectorizer보다 훨씬 풍부한 의미를 담는 벡터를 사용

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

# 문장
texts = [
    "영화가 좋다",
    "영화가 진짜로 좋다"
]

# 1) 문장 → 단어 빈도 벡터
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print("단어 사전:", vectorizer.get_feature_names_out())


# 2) 정규화 (방향만 남기기)
normalizer = Normalizer()
X_norm = normalizer.fit_transform(X)

# 3) 코사인 유사도
sim = cosine_similarity(X_norm[0], X_norm[1])
print("코사인 유사도:", sim[0][0])

단어 사전: ['영화가' '좋다' '진짜로']
코사인 유사도: 0.8164965809277261


### 스케일링 vs 정규화 — 언제 뭘 쓰나?

| 구분 | 스케일링 (Scaling) | 정규화 (Normalization) |
|------|-------------------|------------------------|
| 기준 | **열(Feature)** | **행(Sample)** |
| 바꾸는 것 | 피처의 단위/범위 | 벡터의 길이(L2 norm) |
| 남기는 것 | 피처 간 공정함 | **방향(패턴)** |
| 주 사용처 | 선형/거리 기반 포함 대부분 | 텍스트/추천/임베딩(코사인) |


> **선형/거리 기반 포함 ML에서 기본으로 스케일링을 사용**
>
> 정규화는 "방향"이 중요한 특수한 경우(텍스트/임베팅/추천/등)에만 사용함

---
## Part 4. 데이터 누수 (Data Leakage)

### 데이터 누수란?

- 훈련+시험 전체 데이터를 fit → 시험지 평균/표준편차까지 미리 알게 됨
- 마치 학생이 **시험지를 미리 훔쳐보고 공부한 것**과 같습니다.

> 시험 볼 때는 성적이 뻥튀기되지만, 실제 실력은 아닙니다.  
> Pipeline을 쓰면 "훈련 데이터로만 fit, 시험 데이터는 transform" 규칙을 자동으로 지켜 안전합니다.

### 데이터 누수가 왜 위험한가?

```
연구실에서: "와! 정확도 99%!"
실제 배포 후: "왜 정확도가 80%밖에 안 나오지...?"
```

**누수가 발생하면:**
1. 실험 결과를 **신뢰할 수 없음**
2. 모델 선택이 **잘못될 수 있음**
3. 배포 후 **성능 하락** 발생

> **"전처리는 반드시 훈련셋으로만 fit!"**

In [12]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, stratify=y, random_state=42)

# (나쁜 예시) 전체 데이터로 fit
# 스탠다드스케일러(평균, 표준편차 계산)
# 시험 데이터(X_test) 데이터도 평균/표준편차가 반영이 됨
# 데이터 누수
scaler_bad = StandardScaler().fit(X)
# 위에서 계산된(시험 데이터가 포함된) 평균과 표준편차로, 학습/시험 데이터를 변환
X_train_bad = scaler_bad.transform(X_train)
X_test_bad  = scaler_bad.transform(X_test)
knn_bad = KNeighborsClassifier(n_neighbors=5).fit(X_train_bad, y_train)
pred_bad = knn_bad.predict(X_test_bad)
print(f"[누수 발생·KNN] Acc={accuracy_score(y_test, pred_bad):.3f}, F1(macro)={f1_score(y_test, pred_bad, average='macro'):.3f}")
# [누수 발생·KNN] Acc=0.911, F1(macro)=0.910


[누수 발생·KNN] Acc=0.911, F1(macro)=0.910


In [13]:
from sklearn.pipeline import Pipeline

# (Pipeline) 훈련에만 fit
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
pipe.fit(X_train, y_train)
pred_pipe = pipe.predict(X_test)
print(f"[Pipeline·KNN]  Acc={accuracy_score(y_test, pred_pipe):.3f}, F1(macro)={f1_score(y_test, pred_pipe, average='macro'):.3f}")

[Pipeline·KNN]  Acc=0.911, F1(macro)=0.910


### Pipeline이란?

전처리 + 모델을 **한 줄로 묶어서** 실수를 방지합니다.

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),  # 전처리
    ("model", LogisticRegression()) # 모델
])

pipe.fit(X_train, y_train)   # 훈련셋으로 fit
pipe.predict(X_test)         # 시험셋은 transform만
```

> Pipeline을 쓰면 **데이터 누수가 자동으로 방지**됩니다!

### 로지스틱 회귀(선형 모델)에서도 확인해 보기
- 선형 모델은 스케일링의 영향을 더 많이 받는 편임
- "누수 전처리 vs Pipeline"을 로지스틱 회귀에서도 비교

In [14]:
from sklearn.linear_model import LogisticRegression


# (나쁜 예시) 전체 데이터로 fit한 스케일러 사용
lr_bad = LogisticRegression(max_iter=5000, random_state=42)
lr_bad.fit(X_train_bad, y_train)
pred_lr_bad = lr_bad.predict(X_test_bad)

# (올바른 예시) Pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
pred_lr_pipe = pipe_lr.predict(X_test)

print(f"[누수 발생·LR]  Acc={accuracy_score(y_test, pred_lr_bad):.3f}, F1(macro)={f1_score(y_test, pred_lr_bad, average='macro'):.3f}")
print(f"[Pipeline·LR]   Acc={accuracy_score(y_test, pred_lr_pipe):.3f}, F1(macro)={f1_score(y_test, pred_lr_pipe, average='macro'):.3f}")

[누수 발생·LR]  Acc=0.911, F1(macro)=0.911
[Pipeline·LR]   Acc=0.911, F1(macro)=0.911


### 결과 해석

- Iris처럼 단순한 데이터는 차이가 작아 보일 수 있어도
  **누수 전처리 방식은 원칙적으로 금지**
- 실제 업무 데이터(복잡, 잡음, 범주형 많음)에서는 누수 전처리를 하면
  성능이 **과장되어 보이고**, 배포 후 성능 하락으로 이어질 가능성 큼
- **Pipeline**을 쓰면 전처리 `fit`이 훈련셋에만 적용되고,
  시험셋에는 `transform`만 적용되므로 **항상 안전하고 공정** 함.

> **핵심 요약**
> - 전처리는 반드시 **훈련셋으로만 fit**
> - 시험셋은 반드시 **처음 보는 데이터**처럼 다뤄야 함
> - ColumnTransformer/Pipeline을 습관처럼 사용

### 교차검증으로 누수 vs Pipeline 비교

- train/test split은 한 번만 나누기 때문에 운에 따라 결과가 달라질 수 있음
- 교차검증(K-Fold CV)은 데이터를 여러 번 나눠 평균 성능을 구하기 때문에 훨씬 안정적
- 이번에는 **누수 전처리 방식**과 **Pipeline 방식**을 교차검증으로 비교해봅시다

In [15]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

X, y = load_iris(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# (나쁜 예시) 전체 데이터로 fit → transform 적용
scaler_bad = StandardScaler().fit(X)   # 전체 데이터 사용 → 누수
X_scaled_bad = scaler_bad.transform(X)

knn_bad = KNeighborsClassifier(n_neighbors=5)
scores_bad = cross_val_score(knn_bad, X_scaled_bad, y, cv=cv, scoring="accuracy")

# (올바른 예시) Pipeline (훈련셋만 fit, 시험셋은 transform만)
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])
scores_pipe = cross_val_score(pipe_knn, X, y, cv=cv, scoring="accuracy")

print("[누수 발생·KNN]   mean=%.3f ± %.3f" % (scores_bad.mean(), scores_bad.std()))
print("[Pipeline·KNN]    mean=%.3f ± %.3f" % (scores_pipe.mean(), scores_pipe.std()))
# [누수 발생·KNN]   mean=0.967 ± 0.037
# [Pipeline·KNN]    mean=0.973 ± 0.025


[누수 발생·KNN]   mean=0.967 ± 0.037
[Pipeline·KNN]    mean=0.973 ± 0.025


### 교차검증 결과 해석

- **누수 전처리 방식**: 시험 데이터 정보를 슬쩍 본 상태라 성능이 더 높게 나올 수 있음
- **Pipeline 방식**: 훈련 데이터에만 fit → 시험 데이터에는 transform만 → 더 공정하고 현실적인 성능

> **핵심 정리**
> - 교차검증을 쓰면 "누수 vs Pipeline"의 차이를 평균 성능으로 안정적으로 비교할 수 있다
> - 실제 데이터에서는 누수 방식이 **성능을 뻥튀기**해서 보여주는 경우가 많음
> - **Pipeline**을 습관처럼 사용해야 안전하다

### 로지스틱 회귀(Logistic Regression)로도 비교해보기

- 로지스틱은 스케일링의 영향을 더 많이 받는 모델입니다.
- 이번에는 로지스틱 회귀에서 **누수 vs Pipeline**을 교차검증으로 비교합니다.

In [16]:
from sklearn.linear_model import LogisticRegression

# (나쁜 예시) 누수 스케일링
lr_bad = LogisticRegression(max_iter=5000, random_state=42)
scores_lr_bad = cross_val_score(lr_bad, X_scaled_bad, y, cv=cv, scoring="accuracy")

# (올바른 예시) Pipeline
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000, random_state=42))
])
scores_lr_pipe = cross_val_score(pipe_lr, X, y, cv=cv, scoring="accuracy")

print("[누수 발생·LR]   mean=%.3f ± %.3f" % (scores_lr_bad.mean(), scores_lr_bad.std()))
print("[Pipeline·LR]    mean=%.3f ± %.3f" % (scores_lr_pipe.mean(), scores_lr_pipe.std()))
# [누수 발생·LR]   mean=0.953 ± 0.045
# [Pipeline·LR]    mean=0.953 ± 0.045

[누수 발생·LR]   mean=0.953 ± 0.045
[Pipeline·LR]    mean=0.953 ± 0.045


---
## 오늘의 정리

| 개념 | 핵심 | 코드 |
|------|------|------|
| 인코딩 | 글자→숫자, 순서 없으면 원-핫 | `OneHotEncoder()` |
| 스케일링 | 값 크기 차이 맞추기 | `StandardScaler()`, `MinMaxScaler()` |
| 정규화 | 샘플 길이=1, 방향만 중요할 때 | `Normalizer()` |
| 데이터 누수 | 시험 정보가 훈련에 들어가면 안 됨 | `Pipeline()` |

> **기억할 흐름**: 전처리(인코딩→스케일링) → Pipeline으로 묶기 → 교차검증

---
## 다음 시간 예고

- **평가지표 심화**: 정확도 90%를 믿어도 되나?
- **Confusion Matrix**: TP/FP/FN/TN 읽는 법
- **Precision vs Recall**: 스팸필터 vs 암진단
- **F1, ROC-AUC, PR-AP**: 불균형 데이터 평가

> 오늘 데이터를 준비하는 방법을 익혔으니,  
> 다음엔 **모델 성능을 제대로 평가하는 방법**을 배울거임
